In [1]:
#!/usr/bin/env python3
"""
Script untuk mengambil data iklim dari database TimescaleDB (SKEMA BARU).
- Format observasi: long (source sebagai kolom)
- Metadata stasiun terpisah
- Tanpa qc_flags (karena dihapus sementara)
"""

import pandas as pd
from sqlalchemy import create_engine, text
import yaml
from pathlib import Path
import sys
# ==============================
# 1. KONEKSI DATABASE
# ==============================
# database:
#   host: 172.19.0.201    # ← IP server database
#   port: 5432            # ← Port PostgreSQL default
#   name: climate_db      # ← Nama database
#   user: api             # ← User database
#   password: climate2026 # ← Password (pastikan aman!)

config = {'database': {
        'host': '172.19.0.201',
        'port': 5432,
        'name': 'climate_db',
        'user': 'api',
        'password': 'climate2026'
    }}
def get_db_engine(config_path=None):
    """Buat koneksi database SQLAlchemy."""
    url = f"postgresql://{config['database']['user']}:{config['database']['password']}@" \
          f"{config['database']['host']}:{config['database']['port']}/{config['database']['name']}"
    return create_engine(url)
# ==============================
# 2. FUNGSI UTAMA QUERY (SKEMA BARU)
# ==============================

def query_climate_data(
    parameters=None,
    sources='qc',
    baseline=None,  # ← biarkan None untuk otomatisasi
    min_80pct_only=True,
    min_80pct_baseline='1991',  # default: gunakan 1991 untuk filter
    start_date=None,
    end_date=None,
    stations=None,
    provinces=None,
    regions=None,
    time_aggregation=None,
    spatial_aggregation=None,
    limit=None,
    config_path=None
):
    """
    Ambil data iklim dengan logika otomatis:
    - Jika source='qc', baseline otomatis='1981'
    - Jika min_80pct_only=True, filter berdasarkan kelengkapan di min_80pct_baseline ('1991')
    - Jika min_80pct_baseline='1991', otomatis batasi waktu >= '1991-01-01'
    """
    
    engine = get_db_engine(config_path)
    
    # Normalisasi sources
    if isinstance(sources, str):
        sources = [sources]
    valid_sources = {'raw', 'qc', 'homo', 'extended'}
    if not set(sources).issubset(valid_sources):
        raise ValueError(f"sources harus salah satu dari {valid_sources}")
    
    # 🔑 OTOMATISASI BASELINE
    if baseline is None:
        if 'qc' in sources or sources == ['qc']:
            baseline = '1981'
        else:
            # Untuk homo/extended, biarkan user tentukan atau default ke '1981'
            baseline = '1981'
    
    if baseline not in ['1981', '1991']:
        raise ValueError("baseline harus '1981' atau '1991'")
    
    # Normalisasi parameters
    if parameters is None:
        parameters = ['TEMPERATURE_AVG_C', 'TEMP_24H_TN_C', 'TEMP_24H_TX_C', 'RAINFALL_24H_MM']
    elif isinstance(parameters, str):
        parameters = [parameters]

    # 🔑 OTOMATISASI START_DATE JIKA FILTER BERDASARKAN BASELINE 1991
    effective_start_date = start_date
    if min_80pct_only and min_80pct_baseline == '1991':
        # Pastikan waktu minimal 1991
        if effective_start_date is None or effective_start_date < '1991-01-01':
            effective_start_date = '1991-01-01'

    # Bangun query
    query = """
    SELECT 
        o.time,
        o.wmo_id,
        o.parameter,
        o.source,
        o.value,
        o.baseline,
        o.region,
        m.name,
        m.latitude,
        m.longitude,
        m.province,
        m.regency,
        m.elevation,
        a_obs.availability AS availability,
        a_obs.meets_80pct AS meets_80pct
    FROM observations o
    JOIN station_metadata m ON o.wmo_id = m.wmo_id
    JOIN station_availability a_obs 
        ON o.wmo_id = a_obs.wmo_id 
        AND o.parameter = a_obs.parameter 
        AND o.baseline = a_obs.baseline
    """
    
    # Tambahkan join untuk filter kelengkapan dari baseline lain
    if min_80pct_only:
        query += f"""
        JOIN station_availability a_filter 
            ON o.wmo_id = a_filter.wmo_id 
            AND o.parameter = a_filter.parameter 
            AND a_filter.baseline = '{min_80pct_baseline}'
        """
    
    query += """
    WHERE 
        o.baseline = :baseline
        AND o.parameter = ANY(:parameters)
        AND o.source = ANY(:sources)
    """
    
    params = {
        'baseline': baseline,
        'parameters': parameters,
        'sources': sources
    }
    
    # Filter kelengkapan
    if min_80pct_only:
        query += " AND a_filter.meets_80pct = TRUE"
    
    # Filter waktu (gunakan effective_start_date)
    if effective_start_date:
        query += " AND o.time >= :start_date"
        params['start_date'] = effective_start_date
    if end_date:
        query += " AND o.time <= :end_date"
        params['end_date'] = end_date
    
    # Filter lokasi
    if stations:
        query += " AND o.wmo_id = ANY(:stations)"
        params['stations'] = stations
    if provinces:
        query += " AND m.province = ANY(:provinces)"
        params['provinces'] = provinces
    if regions:
        query += " AND o.region = ANY(:regions)"
        params['regions'] = regions
    
    query += " ORDER BY o.time, o.wmo_id"
    if limit:
        query += " LIMIT :limit"
        params['limit'] = limit

    df = pd.read_sql(text(query), engine, params=params)
    
    # Proses agregasi (sama seperti sebelumnya)
    if time_aggregation and len(df) > 0:
        df['time'] = pd.to_datetime(df['time'])
        if time_aggregation == 'monthly':
            df['time'] = df['time'].dt.to_period('M').dt.start_time
        elif time_aggregation == 'yearly':
            df['time'] = df['time'].dt.to_period('Y').dt.start_time
        
        group_cols = [
            'time', 'wmo_id', 'parameter', 'source', 'baseline', 'region',
            'name', 'latitude', 'longitude', 'province', 'regency', 'elevation'
        ]
        numeric_cols = ['value', 'availability']
        agg_dict = {col: 'mean' for col in numeric_cols}
        df = df.groupby(group_cols).agg(agg_dict).reset_index()
    
    if spatial_aggregation and len(df) > 0:
        df['time'] = pd.to_datetime(df['time'])
        if spatial_aggregation == 'province':
            group_cols = ['time', 'parameter', 'source', 'province']
        elif spatial_aggregation == 'region':
            group_cols = ['time', 'parameter', 'source', 'region']
        elif spatial_aggregation == 'national':
            group_cols = ['time', 'parameter', 'source']
        else:
            group_cols = ['time', 'parameter', 'source']
        
        numeric_cols = ['value', 'availability']
        agg_dict = {col: ['mean', 'std', 'count'] for col in numeric_cols}
        df = df.groupby(group_cols).agg(agg_dict).round(4).reset_index()
        df.columns = ['_'.join(col).strip('_') for col in df.columns.values]
    
    print(f"✅ Mengambil {len(df)} baris data (baseline={baseline}, "
          f"filter kelengkapan dari baseline={min_80pct_baseline}, "
          f"waktu ≥ {effective_start_date})")
    return df

# ==============================
# 3. FUNGSI TAMBAHAN: INFO DATA
# ==============================
def get_data_summary(df):
    """Dapatkan ringkasan informasi tentang data yang diambil."""
    if df.empty:
        print("DataFrame kosong")
        return
    print("\n📊 RINGKASAN DATA:")
    print(f"Total baris: {len(df)}")
    print(f"Stasiun unik: {df['wmo_id'].nunique()}")
    print(f"Parameter: {df['parameter'].unique().tolist()}")
    print(f"Sumber: {df['source'].unique().tolist()}")
    print(f"Periode: {df['time'].min()} s.d. {df['time'].max()}")
    if 'province' in df.columns:
        print(f"Provinsi: {df['province'].nunique()}")
    if 'region' in df.columns:
        print(f"Region: {df['region'].nunique()}")
    if 'meets_80pct' in df.columns:
        good_pct = df['meets_80pct'].mean() * 100
        print(f"Persentase stasiun ≥80%: {good_pct:.1f}%")        

In [ ]:
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta

# 1. Algoritma menentukan akhir bulan sebelumnya (Relative to Today)
# Jika hari ini Feb 2026, maka end_date adalah 31 Jan 2026
today = datetime.now()
last_day_prev_month = (today.replace(day=1) - relativedelta(days=1))
end_date_str = last_day_prev_month.strftime('%Y-%m-%d')

# 2. Eksekusi Query
df = query_climate_data(
    parameters=['TEMPERATURE_AVG_C'],
    sources='homo',
    baseline='1991',
    min_80pct_only=True,
    start_date='1991-01-01',
    end_date=end_date_str  # Menggunakan variabel dinamis
)
# 3. Cek Output
print(f"Periode data hingga: {end_date_str}")
print(f"Jumlah WMO ID: {df[df['parameter'] == 'TEMPERATURE_AVG_C']['wmo_id'].nunique()}")
df.head()

✅ Mengambil 611990 baris data (baseline=1991, filter kelengkapan dari baseline=1991, waktu ≥ 1991-01-01)
Periode data hingga: 2026-01-31
Jumlah WMO ID: 118


,time,wmo_id,parameter,source,value,baseline,region,name,latitude,longitude,province,regency,elevation,availability,meets_80pct
0,1991-01-01,96001,TEMPERATURE_AVG_C,homo,26.65,1991,2,Stasiun Meteorologi Maimun Saleh,5.87655,95.33785,Nanggroe Aceh Darussalam,Kota Sabang,126.0,99.937584,True
1,1991-01-01,96011,TEMPERATURE_AVG_C,homo,26.45,1991,2,Stasiun Meteorologi Sultan Iskandar Muda,5.52244,95.41700,Nanggroe Aceh Darussalam,Kab. Aceh Besar,20.0,97.433100,True
2,1991-01-01,96015,TEMPERATURE_AVG_C,homo,26.20,1991,2,Stasiun Meteorologi Cut Nyak Dhien Nagan Raya,4.04928,96.24796,Nanggroe Aceh Darussalam,Kab. Nagan Raya,3.0,98.572210,True
3,1991-01-01,96031,TEMPERATURE_AVG_C,homo,26.10,1991,2,Stasiun Klimatologi Sumatera Utara,3.62114,98.71485,Sumatera Utara,Kab. Deli Serdang,25.0,99.836160,True
4,1991-01-01,96033,TEMPERATURE_AVG_C,homo,27.20,1991,2,Stasiun Meteorologi Maritim Belawan,3.78824,98.71492,Sumatera Utara,Kota Medan,3.0,99.446045,True


**CONVERT DATA TO WIDH FORMAT**

In [ ]:
df = pd.read_csv('/mnt/dataset/02_REPO_GITHUB_FIRMAN/Developing_Climate_Observation_Dataset/data/04.Dataset_Final/TEMPERATURE_AVG_C_homogen_final_baseline_1991.csv')
DF_ROBI = df[['WMO_ID','DATA_TIMESTAMP', 
            'HOMO_TEMPERATURE_AVG_C']]
# 1. Pastikan kolom waktu dalam format datetime
DF_ROBI['DATA_TIMESTAMP'] = pd.to_datetime(DF_ROBI['DATA_TIMESTAMP'])

# 2. Filter data (Sesudah 1 Des 2025)
DF_ROBI = DF_ROBI[DF_ROBI['DATA_TIMESTAMP'] > '2025-12-01'].copy()

# 3. Ekstrak Thn, Bln, Tgl
DF_ROBI['Thn'] = DF_ROBI['DATA_TIMESTAMP'].dt.year
DF_ROBI['Bln'] = DF_ROBI['DATA_TIMESTAMP'].dt.month
DF_ROBI['Tgl'] = DF_ROBI['DATA_TIMESTAMP'].dt.day

# 4. Transformasi ke format WIDE
# index: Baris identitas waktu
# columns: Nama stasiun (WMO_ID) akan menjadi header kolom
# values: Data suhu yang ingin ditampilkan
DF_WIDE = DF_ROBI.pivot_table(
    index=['Thn', 'Bln', 'Tgl'], 
    columns='WMO_ID', 
    values='HOMO_TEMPERATURE_AVG_C'
).reset_index()

# 5. Opsional: Hilangkan nama hirarki pada kolom (supaya bersih)
DF_WIDE.columns.name = None

# Menampilkan hasil
DF_WIDE.to_csv('DES_JAN.csv', index=False)

/tmp/ipykernel_1808438/2673436732.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DF_ROBI['DATA_TIMESTAMP'] = pd.to_datetime(DF_ROBI['DATA_TIMESTAMP'])


In [61]:
import pandas as pd

# 1. Seleksi dan Persiapan Data
# Menyesuaikan dengan kolom yang Anda definisikan di awal
df_sel = df[['WMO_ID', 'NAME', 'CURRENT_LATITUDE', 'CURRENT_LONGITUDE', 
            'PROVINSI', 'KABUPATEN', 'ELEVATION', 'DATA_TIMESTAMP', 
            'HOMO_TEMPERATURE_AVG_C']].copy()

# Rename DATA_TIMESTAMP ke 'time' untuk memudahkan grouping
df_sel['time'] = pd.to_datetime(df_sel['DATA_TIMESTAMP'])

# 2. Agregasi Bulanan
# Menggunakan 'first' untuk kolom meta-data agar tidak hilang setelah grouping
df_sel_monthly = df_sel.groupby(['WMO_ID', pd.Grouper(key='time', freq='ME')]).agg(
    name=('NAME', 'first'),
    latitude=('CURRENT_LATITUDE', 'first'),
    longitude=('CURRENT_LONGITUDE', 'first'),
    province=('PROVINSI', 'first'),
    regency=('KABUPATEN', 'first'),
    elevation=('ELEVATION', 'first'),
    value=('HOMO_TEMPERATURE_AVG_C', 'mean'),
    days_present=('HOMO_TEMPERATURE_AVG_C', 'count')
).reset_index()

# 3. Hitung Parameter Kelayakan (Threshold 80%)
df_sel_monthly['days_in_month'] = df_sel_monthly['time'].dt.daysinmonth
df_sel_monthly['data_completeness'] = (df_sel_monthly['days_present'] / df_sel_monthly['days_in_month']) * 100

# Standar: Hanya anggap valid jika ketersediaan data > 80%
df_sel_monthly['is_valid_for_normal'] = df_sel_monthly['data_completeness'] > 80

# 4. Hitung Nilai Normal (Klimatologi)
df_sel_monthly['month'] = df_sel_monthly['time'].dt.month
climatology = (
    df_sel_monthly[df_sel_monthly['is_valid_for_normal']]
    .groupby(['WMO_ID', 'month'])['value']
    .mean()
    .reset_index()
)
climatology.rename(columns={'value': 'normal'}, inplace=True)

# 5. Hitung Anomali
df_sel_final = pd.merge(df_sel_monthly, climatology, on=['WMO_ID', 'month'], how='left')
df_sel_final['anomali'] = df_sel_final['value'] - df_sel_final['normal']

# 6. Pemeringkatan dan Selisih (Ranking & Diff)
df_sel_final = df_sel_final.sort_values(['WMO_ID', 'time'])
df_sel_final['year'] = df_sel_final['time'].dt.year

# Selisih anomali dari bulan sebelumnya untuk stasiun yang sama
df_sel_final['anomali_diff'] = df_sel_final.groupby('WMO_ID')['anomali'].diff()

# Ranking: Semakin tinggi anomali (semakin panas), ranking semakin kecil (Ranking 1 = Terpanas)
df_sel_final['rank_from_all_station'] = df_sel_final.groupby(['year', 'month'])['anomali'].rank(ascending=False, method='min')

# Ranking historis untuk stasiun tersebut di bulan yang sama (misal: Jan 2026 dibanding Jan 2025, 2024...)
df_sel_final['rank_from_all_month_history'] = df_sel_final.groupby(['WMO_ID', 'month'])['anomali'].rank(ascending=False, method='min')

# Menampilkan hasil
print(df_sel_final[['WMO_ID', 'name', 'time', 'value', 'normal', 'anomali', 'rank_from_all_station']].head())

KeyError: "None of [Index(['WMO_ID', 'NAME', 'CURRENT_LATITUDE', 'CURRENT_LONGITUDE', 'PROVINSI',\n       'KABUPATEN', 'ELEVATION', 'DATA_TIMESTAMP', 'HOMO_TEMPERATURE_AVG_C'],\n      dtype='object')] are in the [columns]"

In [42]:
df_sel_final.to_csv('01.Testing_Anomali_Dataset.csv', index=False)

In [45]:
df_this_month = df_sel_final[(df_sel_final['time'] > pd.Timestamp('2026-01-01')) & (df_sel_final['time'] <= pd.Timestamp('2026-01-31')) & df_sel_final['is_valid_for_normal']]
len(df_this_month[df_this_month['is_valid_for_normal']])

116

In [46]:
#rangking 5 teratas
df_this_month = df_sel_final[(df_sel_final['time'] > pd.Timestamp('2026-01-01')) & (df_sel_final['time'] <= pd.Timestamp('2026-01-31'))]
top_5_ranking_teratas = df_this_month.nsmallest(5, 'rank_from_all_station')
top_5_ranking_terbawah = df_this_month.nlargest(5, 'rank_from_all_station')
print(top_5_ranking_teratas[['WMO_ID', 'name', 'anomali', 'rank_from_all_station']])
print(top_5_ranking_terbawah[['WMO_ID', 'name', 'anomali', 'rank_from_all_station']])

       WMO_ID                                   name   anomali  \
37747   97192        Stasiun Meteorologi Beto Ambari  1.099634   
43816   97630              Stasiun Meteorologi Torea  1.059073   
19466   96655       Stasiun Meteorologi Tjilik Riwut  1.008714   
43435   97600            Stasiun Meteorologi Emalamo  0.986293   
47845   97810  Stasiun Meteorologi Karel Sadsuitubun  0.913999   

       rank_from_all_station  
37747                    1.0  
43816                    2.0  
19466                    3.0  
43435                    4.0  
47845                    5.0  
       WMO_ID                                       name   anomali  \
24434   96753             Stasiun Klimatologi Jawa Barat -0.838516   
22386   96741  Stasiun Meteorologi Maritim Tanjung Priok -0.667769   
22808   96745              Stasiun Meteorologi Kemayoran -0.646084   
23591   96749         Stasiun Meteorologi Soekarno Hatta -0.601722   
24013   96751                 Stasiun Meteorologi Citeko -0.597805 

In [27]:
# 1. Menghitung jumlah stasiun yang memiliki nilai normal per bulan
# Kita menggunakan tabel 'climatology' karena tabel ini hanya berisi data yang lolos sensor 80%
summary_normal = climatology.groupby('month')['WMO_ID'].count().reset_index()
summary_normal.columns = ['Bulan', 'Jumlah_Stasiun_Valid']

# 2. Menghitung persentase stasiun yang memiliki nilai normal dibanding total stasiun unik
total_stasiun_unik = df['WMO_ID'].nunique()
summary_normal['persentase_cakupan'] = (summary_normal['Jumlah_Stasiun_Valid'] / total_stasiun_unik) * 100

# 3. Opsional: Tambahkan nama bulan agar lebih mudah dibaca
import calendar
summary_normal['Nama_Bulan'] = summary_normal['Bulan'].apply(lambda x: calendar.month_name[x])

# Reorder kolom untuk kerapihan
summary_normal = summary_normal[['Bulan', 'Nama_Bulan', 'Jumlah_Stasiun_Valid', 'persentase_cakupan']]

print(f"Total stasiun unik dalam dataset: {total_stasiun_unik}")
print(summary_normal)

Total stasiun unik dalam dataset: 118
    Bulan Nama_Bulan  Jumlah_Stasiun_Valid  persentase_cakupan
0       1    January                   118               100.0
1       2   February                   118               100.0
2       3      March                   118               100.0
3       4      April                   118               100.0
4       5        May                   118               100.0
5       6       June                   118               100.0
6       7       July                   118               100.0
7       8     August                   118               100.0
8       9  September                   118               100.0
9      10    October                   118               100.0
10     11   November                   118               100.0
11     12   December                   118               100.0


**Melakukan Perhitungan Anomali Suhu**

In [64]:
import pandas as pd

# 1. Persiapan Data
df['time'] = pd.to_datetime(df['time'])

# 2. Agregasi Bulanan
df_monthly = df.groupby(['wmo_id', pd.Grouper(key='time', freq='ME')]).agg(
    parameter=('parameter', 'first'),
    source=('source', 'first'),
    baseline=('baseline', 'first'),
    region=('region', 'first'),
    name=('name', 'first'),
    latitude=('latitude', 'first'),
    longitude=('longitude', 'first'),
    province=('province', 'first'),
    regency=('regency', 'first'),
    elevation=('elevation', 'first'),
    value=('value', 'mean'),
    days_present=('value', 'count')
).reset_index()

# 3. Hitung Parameter Kelayakan (Threshold 80%)
df_monthly['days_in_month'] = df_monthly['time'].dt.daysinmonth
df_monthly['data_completeness'] = (df_monthly['days_present'] / df_monthly['days_in_month']) * 100

# Standar: Hanya anggap valid jika ketersediaan data > 80%
df_monthly['is_valid_for_normal'] = df_monthly['data_completeness'] > 80

# 4. Hitung Nilai Normal (Klimatologi)
# FILTER: Hanya stasiun & bulan yang ketersediaannya > 80% yang dihitung
df_monthly['month'] = df_monthly['time'].dt.month
climatology = (
    df_monthly[df_monthly['is_valid_for_normal']]
    .groupby(['wmo_id', 'month'])['value']
    .mean()
    .reset_index()
)
climatology.rename(columns={'value': 'normal'}, inplace=True)

# 5. Hitung Anomali
# Gabungkan kembali ke data bulanan utama
df_final = pd.merge(df_monthly, climatology, on=['wmo_id', 'month'], how='left')

# Anomali = Suhu Aktual - Nilai Normal
df_final['anomali'] = df_final['value'] - df_final['normal']

# 6. Hasil Akhir (Mengurutkan agar enak dibaca)
df_final = df_final.sort_values(['wmo_id', 'time'])
df_final['anomali_diff'] = df_final.groupby('wmo_id')['anomali'].diff()
df_final['year'] = df_final['time'].dt.year
df_final['rank_from_all_station'] = df_final.groupby(['year', 'month'])['anomali'].rank(ascending=False, method='min')
df_final['rank_from_all_month'] = df_final.groupby(['wmo_id','month'])['anomali'].rank(ascending=False, method='min')

In [65]:
# 1. Menghitung jumlah stasiun yang memiliki nilai normal per bulan
# Kita menggunakan tabel 'climatology' karena tabel ini hanya berisi data yang lolos sensor 80%
summary_normal = climatology.groupby('month')['wmo_id'].count().reset_index()
summary_normal.columns = ['Bulan', 'Jumlah_Stasiun_Valid']

# 2. Menghitung persentase stasiun yang memiliki nilai normal dibanding total stasiun unik
total_stasiun_unik = df['wmo_id'].nunique()
summary_normal['persentase_cakupan'] = (summary_normal['Jumlah_Stasiun_Valid'] / total_stasiun_unik) * 100

# 3. Opsional: Tambahkan nama bulan agar lebih mudah dibaca
import calendar
summary_normal['Nama_Bulan'] = summary_normal['Bulan'].apply(lambda x: calendar.month_name[x])

# Reorder kolom untuk kerapihan
summary_normal = summary_normal[['Bulan', 'Nama_Bulan', 'Jumlah_Stasiun_Valid', 'persentase_cakupan']]

print(f"Total stasiun unik dalam dataset: {total_stasiun_unik}")
print(summary_normal)

Total stasiun unik dalam dataset: 118
    Bulan Nama_Bulan  Jumlah_Stasiun_Valid  persentase_cakupan
0       1    January                   118          100.000000
1       2   February                   117           99.152542
2       3      March                   117           99.152542
3       4      April                   113           95.762712
4       5        May                   118          100.000000
5       6       June                   116           98.305085
6       7       July                   118          100.000000
7       8     August                   117           99.152542
8       9  September                   113           95.762712
9      10    October                   116           98.305085
10     11   November                   114           96.610169
11     12   December                   118          100.000000


In [68]:
#rangking 5 teratas
df_this_month = df_final[(df_final['time'] > pd.Timestamp('2026-01-01')) & (df_final['time'] <= pd.Timestamp('2026-01-31'))]
top_5_ranking_teratas = df_this_month.nsmallest(5, 'rank_from_all_station')
top_5_ranking_terbawah = df_this_month.nlargest(5, 'rank_from_all_station')
print(top_5_ranking_teratas[['wmo_id', 'name', 'anomali', 'rank_from_all_station']])
print(top_5_ranking_terbawah[['wmo_id', 'name', 'anomali', 'rank_from_all_station']])

      wmo_id                                name   anomali  \
15098  96557      Stasiun Meteorologi Nangapinoh  2.578167   
19160  96655    Stasiun Meteorologi Tjilik Riwut  2.343250   
20379  96733          Stasiun Klimatologi Banten  1.554161   
42280  97560  Stasiun Meteorologi Frans Kaisiepo  1.369756   
17941  96615   Stasiun Meteorologi Rahadi Oesman  1.364429   

       rank_from_all_station  
15098                    1.0  
19160                    2.0  
20379                    3.0  
42280                    4.0  
17941                    5.0  
      wmo_id                                       name   anomali  \
22026  96741  Stasiun Meteorologi Maritim Tanjung Priok -1.374528   
21611  96739               Stasiun Meteorologi Budiarto -1.173775   
29415  96945                 Stasiun Geofisika Pasuruan -1.128175   
26523  96837   Stasiun Meteorologi Maritim Tanjung Emas -0.999546   
30236  96973              Stasiun Meteorologi Trunojoyo -0.888744   

       rank_from_all_stati